In [1]:
!pip install transformers sentencepiece
!pip install torch

In [2]:
from transformers import pipeline

sentiment_model = pipeline("sentiment-analysis")
emotion_model = pipeline("text-classification",
                         model="j-hartmann/emotion-english-distilroberta-base",
                         return_all_scores=False)

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/329M [00:00<?, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Device set to use cuda:0
/usr/local/lib/python3.12/dist-packages/transformers/pipelines/text_classification.py:111: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


In [3]:
!python -m spacy download en_core_web_trf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.4/457.4 MB 729.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 237.9/237.9 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 734.0/734.0 kB 43.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_trf')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [5]:
import json
import spacy
from transformers import pipeline
from collections import Counter

# Load spaCy large English model
nlp = spacy.load("en_core_web_trf")

# Load sentiment classifier
sentiment_classifier = pipeline("sentiment-analysis", device=0)

# Function to extract persona from ESConv sample
def extract_persona_esconv(sample):


    #   1. Extract seeker text
    dialog = sample.get("dialog", [])
    seeker_utts = [turn["content"] for turn in dialog if turn.get("speaker") == "seeker"]

    full_text = " ".join(seeker_utts)

    # Run spaCy
    doc = nlp(full_text)


    #   2. Named Entities
    entities = {}
    for ent in doc.ents:
        if ent.label_ in ["PERSON", "ORG", "GPE", "DATE", "AGE"]:
            entities.setdefault(ent.label_, []).append(ent.text)


    #   3. Main Nouns
    nouns = [token.text for token in doc if token.pos_ in ["NOUN", "PROPN"]]
    noun_counts = Counter(nouns)
    main_nouns = [w for w, c in noun_counts.most_common(30)]


    #   4. Personality Traits
    adjectives = [token.text for token in doc if token.pos_ == "ADJ"]
    adj_counts = Counter(adjectives)
    personality_traits = [w for w, c in adj_counts.most_common(20)]


    #   5. Key Sentences
    sentences = [sent.text.strip() for sent in doc.sents if len(sent.text.strip()) > 10]
    key_sentences = sentences[:8]


    #   6. Sentiment
    if len(full_text) > 0:
        sentiment_result = sentiment_classifier(full_text[:512])[0]
        sentiment = sentiment_result["label"]
    else:
        sentiment = "NEUTRAL"


    #   7. Coping Style
    coping_style = []
    if "don't know" in full_text or "not sure" in full_text:
        coping_style.append("uncertain decision-making")
    if "talk" in full_text or "communicate" in full_text:
        coping_style.append("prefers communication when ready")


    #   8. Values
    values = []
    for v in ["communication", "relationships", "support", "family", "friendship", "career"]:
        if v in full_text.lower():
            values.append(v)


    #   9. Emotions
    emotion = sample.get("emotion_type", "neutral")


    #   10. Combine Persona
    persona = {
        "Experience Type": sample.get("experience_type"),
        "Problem Type": sample.get("problem_type"),
        "Given Emotion Type": sample.get("emotion_type"),
        "Situation Summary": sample.get("situation"),

        "Entities": entities,
        "Main Nouns": main_nouns,
        "Personality Traits": personality_traits,
        "Key Sentences": key_sentences,

        "Sentiment": sentiment,
        "Emotion Pattern": emotion,
        "Coping Style": coping_style,
        "Values": values,

        "Survey Score": sample.get("survey_score", {}),
    }

    return persona


#   Load ESConv Dataset
dataset_path = "/content/ESConv.json"

with open(dataset_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# Extract persona from sample number 0
first_sample = data[0]
persona = extract_persona_esconv(first_sample)

import pprint
pprint.pprint(persona)


No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cuda:0


{'Coping Style': [],
 'Emotion Pattern': 'anxiety',
 'Entities': {'DATE': ['every year']},
 'Experience Type': 'Previous Experience',
 'Given Emotion Type': 'anxiety',
 'Key Sentences': ['Hello\n'
                   ' I am having a lot of anxiety about quitting my current '
                   'job.',
                   'It is too stressful but pays well\n'
                   ' I have to deal with many people in hard financial '
                   'situations and it is upsetting \n'
                   ' I do, but often they are not going to get back to what '
                   'they want.',
                   'Many people are going to lose their home when safeguards '
                   'are lifted \n'
                   ' That is true but sometimes I feel like I should put my '
                   'feelings and health first \n'
                   ' Probably not.',
                   'I was with the same company for a long time and I '
                   'consistently get a bonus every 

In [6]:
import json
import spacy
from transformers import pipeline
from collections import Counter

# Load spaCy model
nlp = spacy.load("en_core_web_trf")

# Load sentiment classifier
sentiment_classifier = pipeline("sentiment-analysis", device=0)

def extract_persona_esconv(sample):

    # --- Extract only seeker messages ---
    dialog = sample.get("dialog", [])
    seeker_texts = [turn["content"] for turn in dialog if turn.get("speaker") == "seeker"]
    full_text = " ".join(seeker_texts).strip()

    if not full_text:
        return {
            "User Name": "Unknown",
            "Age": "Unknown",
            "Coping Style": [],
            "Emotion Pattern": sample.get("emotion_type", "neutral"),
            "Entities": {},
            "Key Sentences": [],
            "Main Nouns": [],
            "Personality Traits": [],
            "Sentiment": "neutral",
            "Values": []
        }

    doc = nlp(full_text)

    # --- Entities ---
    entities = {}
    for ent in doc.ents:
        if ent.label_ in ["PERSON", "ORG", "GPE", "LOC", "DATE", "AGE", "NORP"]:
            entities.setdefault(ent.label_, []).append(ent.text)

    # --- Main nouns ---
    nouns = [t.text.lower() for t in doc if t.pos_ in ["NOUN", "PROPN"]]
    noun_counts = Counter(nouns)
    main_nouns = [w for w, c in noun_counts.most_common(30)]

    # --- Personality traits ---
    adjectives = [t.text.lower() for t in doc if t.pos_ == "ADJ"]
    adj_counts = Counter(adjectives)
    personality_traits = [w for w, c in adj_counts.most_common(20)]

    # --- Key sentences ---
    sentences = [sent.text.strip() for sent in doc.sents if len(sent.text.strip()) > 10]
    key_sentences = sorted(sentences, key=lambda s: len(s.split()), reverse=True)[:8]

    # --- Sentiment ---
    try:
        sentiment = sentiment_classifier(full_text[:512])[0]["label"]
    except:
        sentiment = "neutral"

    # --- Coping style ---
    coping_style = []
    txt = full_text.lower()

    if any(x in txt for x in ["don't know", "not sure", "uncertain", "confused"]):
        coping_style.append("uncertain decision-making")
    if any(x in txt for x in ["talk", "communicate", "share feelings"]):
        coping_style.append("prefers communication when ready")
    if any(x in txt for x in ["wait", "avoid", "hold back"]):
        coping_style.append("avoidance")
    if any(x in txt for x in ["think", "reflect", "consider"]):
        coping_style.append("reflective")
    if any(x in txt for x in ["care", "support", "understand"]):
        coping_style.append("empathetic")

    # --- Values ---
    values = []
    value_keywords = [
        "communication", "relationships", "support", "family", "friendship",
        "career", "trust", "honesty", "wellbeing", "help", "learning"
    ]
    for v in value_keywords:
        if v in txt:
            values.append(v)

    # --- Emotion type from dataset ---
    emotion = sample.get("emotion_type", "neutral")

    # --- Assemble persona ---
    persona = {
        "User Name": entities.get("PERSON", ["Unknown"])[0],
        "Age": entities.get("AGE", ["Unknown"])[0],
        "Coping Style": coping_style,
        "Emotion Pattern": emotion,
        "Entities": entities,
        "Key Sentences": key_sentences,
        "Main Nouns": main_nouns,
        "Personality Traits": personality_traits,
        "Sentiment": sentiment,
        "Values": values,
        "Problem Type": sample.get("problem_type", ""),
        "Experience Type": sample.get("experience_type", ""),
        "Situation Summary": sample.get("situation", "")
    }

    return persona


# -------------------------
# Load ESConv dataset
# -------------------------
dataset_path = "/content/ESConv.json"
with open(dataset_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# -------------------------
# Extract persona for first sample
# -------------------------
sample = data[0]
persona = extract_persona_esconv(sample)

print(persona)


No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cuda:0


{'User Name': 'Unknown', 'Age': 'Unknown', 'Coping Style': ['reflective'], 'Emotion Pattern': 'anxiety', 'Entities': {'DATE': ['every year']}, 'Key Sentences': ['It is too stressful but pays well\n I have to deal with many people in hard financial situations and it is upsetting \n I do, but often they are not going to get back to what they want.', 'Many people are going to lose their home when safeguards are lifted \n That is true but sometimes I feel like I should put my feelings and health first \n Probably not.', 'Maybe I just need to sit down and really think about it \n It really is a big decision \n Thank you for the different perspective \n That is true.', 'I was with the same company for a long time and I consistently get a bonus every year  I could try.', 'It mostly gets to me at the end of the day \n That is also true.', 'Hello\n I am having a lot of anxiety about quitting my current job.', 'Sometimes I wonder if it really is for me though  \n That is true.', 'Thanks again'],

In [7]:
import json
import spacy
from transformers import pipeline
from collections import Counter
from tqdm import tqdm

# Load spaCy large English model for NER and POS
nlp = spacy.load("en_core_web_trf")

# Load sentiment classifier
sentiment_classifier = pipeline("sentiment-analysis", device=0)


def extract_persona(sample):

    # Extract only seeker messages
    dialog = sample.get("dialog", [])
    seeker_texts = [turn["content"] for turn in dialog if turn.get("speaker") == "seeker"]
    full_text = " ".join(seeker_texts).strip()

    if not full_text:
        return {
            "User Name": "Unknown",
            "Age": "Unknown",
            "Coping Style": [],
            "Emotion Pattern": sample.get("emotion_type", "neutral"),
            "Entities": {},
            "Key Sentences": [],
            "Main Nouns": [],
            "Personality Traits": [],
            "Sentiment": "neutral",
            "Values": []
        }

    # Run NLP
    doc = nlp(full_text)

    # Extract entities
    entities = {}
    for ent in doc.ents:
        if ent.label_ in ["PERSON", "ORG", "GPE", "LOC", "DATE", "AGE", "NORP"]:
            entities.setdefault(ent.label_, []).append(ent.text)

    # Extract main nouns / proper nouns
    nouns = [token.text.lower() for token in doc if token.pos_ in ["NOUN", "PROPN"]]
    noun_counts = Counter(nouns)
    main_nouns = [word for word, count in noun_counts.most_common(30)]

    # Extract personality traits (adjectives)
    adjectives = [token.text.lower() for token in doc if token.pos_ == "ADJ"]
    adj_counts = Counter(adjectives)
    personality_traits = [word for word, count in adj_counts.most_common(20)]

    # Key sentences
    sentences = [sent.text.strip() for sent in doc.sents if len(sent.text.strip()) > 10]
    sentences_sorted = sorted(sentences, key=lambda x: len(x.split()), reverse=True)
    key_sentences = sentences_sorted[:8]


    # Sentiment
    try:
        sentiment_result = sentiment_classifier(full_text[:512])[0]
        sentiment = sentiment_result["label"]
    except:
        sentiment = "neutral"


    # Coping style heuristics
    coping_style = []
    text_lower = full_text.lower()

    if any(kw in text_lower for kw in ["don't know", "not sure", "uncertain", "confused"]):
        coping_style.append("uncertain decision-making")
    if any(kw in text_lower for kw in ["talk", "communicate", "share feelings"]):
        coping_style.append("prefers communication when ready")
    if any(kw in text_lower for kw in ["wait", "avoid", "hold back"]):
        coping_style.append("avoidance")
    if any(kw in text_lower for kw in ["think", "reflect", "consider"]):
        coping_style.append("reflective")
    if any(kw in text_lower for kw in ["care", "support", "understand"]):
        coping_style.append("empathetic")

    # Values
    values = []
    value_keywords = [
        "communication", "relationships", "support", "family", "friendship",
        "career", "trust", "honesty", "wellbeing", "help", "learning"
    ]

    for v in value_keywords:
        if v in text_lower:
            values.append(v)

    # Emotion Pattern

    emotion = sample.get("emotion_type", "neutral")


    # Assemble persona
    persona = {
        "User Name": entities.get("PERSON", ["Unknown"])[0],
        "Age": entities.get("AGE", ["Unknown"])[0],
        "Coping Style": coping_style,
        "Emotion Pattern": emotion,
        "Entities": entities,
        "Key Sentences": key_sentences,
        "Main Nouns": main_nouns,
        "Personality Traits": personality_traits,
        "Sentiment": sentiment,
        "Values": values,
        "Problem Type": sample.get("problem_type", ""),
        "Experience Type": sample.get("experience_type", ""),
        "Situation Summary": sample.get("situation", "")
    }

    return persona


# Load ESConv dataset
dataset_path = "/content/ESConv.json"

with open(dataset_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# Process entire dataset
all_personas = []
for sample in tqdm(data, desc="Generating ESConv personas"):
    persona = extract_persona(sample)
    all_personas.append(persona)


# Save output

output_path = "/content/PESConv.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(all_personas, f, ensure_ascii=False, indent=4)

print(f"Saved {len(all_personas)} personas to {output_path}")


No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cuda:0
Generating ESConv personas: 100%|██████████| 1300/1300 [18:09<00:00,  1.19it/s]

Saved 1300 personas to /content/PESConv.json
